# Export Hotel Pairs to CSV

This notebook reads the complete hotel pairs data from the Delta Lake path and exports it to a single CSV file for external analysis.

In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    avg,
    round as spark_round,
    min as spark_min,
    max as spark_max,
)
import os

# Override any problematic environment variables
os.environ.pop("HADOOP_CONF_DIR", None)
os.environ.pop("YARN_CONF_DIR", None)

# Create Spark session with MinIO/S3A configuration
# Simplified without Delta Lake dependencies - read as parquet directly
spark = (
    SparkSession.builder.appName("Analyze Hotel Pairs")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000")
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000")
    .config("spark.hadoop.fs.s3a.socket.send.buffer", "8192")
    .config("spark.hadoop.fs.s3a.socket.recv.buffer", "8192")
    .config("spark.hadoop.fs.s3a.attempts.maximum", "3")
    .config("spark.hadoop.fs.s3a.retry.limit", "3")
    .config("spark.hadoop.fs.s3a.retry.interval", "500")
    .config("spark.hadoop.fs.s3a.retry.throttle.limit", "3")
    .config("spark.hadoop.fs.s3a.retry.throttle.interval", "1000")
    .config("spark.hadoop.fs.s3a.connection.maximum", "50")
    .config("spark.hadoop.fs.s3a.threads.max", "10")
    .config("spark.hadoop.fs.s3a.threads.core", "5")
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60")
    .config("spark.hadoop.fs.s3a.max.total.tasks", "5")
    .config("spark.hadoop.fs.s3a.readahead.range", "65536")
    .config("spark.hadoop.fs.s3a.paging.maximum", "5")
    .config("spark.hadoop.fs.s3a.list.version", "2")
    .config("spark.hadoop.fs.s3a.committer.threads", "4")
    .config("spark.hadoop.fs.s3a.committer.staging.tmp.path", "/tmp/staging")
    .config("spark.hadoop.fs.s3a.buffer.dir", "/tmp")
    .config("spark.hadoop.fs.s3a.multipart.size", "104857600")
    .config("spark.hadoop.fs.s3a.multipart.threshold", "2147483647")
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "disk")
    .config("spark.hadoop.fs.s3a.fast.upload.active.blocks", "4")
    .config("spark.hadoop.fs.s3a.block.size", "33554432")
    .config("spark.hadoop.fs.s3a.metadatastore.authoritative", "false")
    .config("spark.sql.files.maxPartitionBytes", "134217728")
    .config("spark.driver.memory", "2g")
    .config("spark.ui.port", "4060")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✓ Spark session created successfully")
print("✓ Spark UI available at: http://localhost:4060")

✓ Spark session created successfully
✓ Spark UI available at: http://localhost:4060


In [25]:
# Read hotel pairs data from the Parquet source
delta_path = "s3a://delta-lake/bronze/hotel_pairs/"
print(f"Reading data from: {delta_path}")

df = spark.read.format("parquet").load(delta_path)

print(
    f"✓ Data loaded successfully with {df.count():,} rows and {len(df.columns)} columns."
)

Reading data from: s3a://delta-lake/bronze/hotel_pairs/
✓ Data loaded successfully with 27,266 rows and 40 columns.


In [26]:
df.printSchema()

root
 |-- id_i: string (nullable = true)
 |-- id_j: string (nullable = true)
 |-- uid_i: string (nullable = true)
 |-- uid_j: string (nullable = true)
 |-- providerName_i: string (nullable = true)
 |-- providerName_j: string (nullable = true)
 |-- providerHotelId_i: string (nullable = true)
 |-- providerHotelId_j: string (nullable = true)
 |-- name_i: string (nullable = true)
 |-- name_j: string (nullable = true)
 |-- normalized_name_i: string (nullable = true)
 |-- normalized_name_j: string (nullable = true)
 |-- type_i: string (nullable = true)
 |-- type_j: string (nullable = true)
 |-- geo_distance_km: double (nullable = true)
 |-- name_score_containment: float (nullable = true)
 |-- normalized_name_score_containment: float (nullable = true)
 |-- name_score_jaccard: float (nullable = true)
 |-- normalized_name_score_jaccard: float (nullable = true)
 |-- name_score_lcs: float (nullable = true)
 |-- normalized_name_score_lcs: float (nullable = true)
 |-- name_score_levenshtein: float 

In [27]:
# Extra columns to join from hotels master table
EXTRA_COLS = [
    "contact_website",
    "starRating",
    "contact_address_line1",
    "contact_address_postalCode",
    "contact_address_city_name",
    "contact_address_state_name",
]

# Check if any of the _i / _j versions are missing
missing = [
    c for c in EXTRA_COLS if f"{c}_i" not in df.columns or f"{c}_j" not in df.columns
]

if missing:
    hotels_path = "s3a://delta-lake/bronze/hotels/"
    print(f"Reading hotel master data from: {hotels_path}")
    hotels = spark.read.format("parquet").load(hotels_path)

    # Only select columns that actually exist in the hotels parquet
    available = [c for c in missing if c in hotels.columns]
    skipped = [c for c in missing if c not in hotels.columns]
    if skipped:
        print(f"⚠ Skipping columns not found in hotels parquet: {skipped}")

    hotels_slim = hotels.select("providerHotelId", *available)

    # Join _i side
    hotels_i = hotels_slim
    for c in available:
        hotels_i = hotels_i.withColumnRenamed(c, f"{c}_i")
    hotels_i = hotels_i.withColumnRenamed("providerHotelId", "_join_i")
    df = df.join(hotels_i, df["providerHotelId_i"] == col("_join_i"), how="left").drop(
        "_join_i"
    )

    # Join _j side
    hotels_j = hotels_slim
    for c in available:
        hotels_j = hotels_j.withColumnRenamed(c, f"{c}_j")
    hotels_j = hotels_j.withColumnRenamed("providerHotelId", "_join_j")
    df = df.join(hotels_j, df["providerHotelId_j"] == col("_join_j"), how="left").drop(
        "_join_j"
    )

    print(f"✓ Joined {available} for both _i and _j. Total columns: {len(df.columns)}")
else:
    print("✓ All extra columns already present in parquet — skipping join")

Reading hotel master data from: s3a://delta-lake/bronze/hotels/
⚠ Skipping columns not found in hotels parquet: ['contact_website']
✓ Joined ['starRating', 'contact_address_line1', 'contact_address_postalCode', 'contact_address_city_name', 'contact_address_state_name'] for both _i and _j. Total columns: 50


In [28]:
hotels_path = "s3a://delta-lake/bronze/hotels/"
hotels = spark.read.format("parquet").load(hotels_path)
print(f"Hotels schema ({len(hotels.columns)} columns):")
hotels.printSchema()

Hotels schema (35 columns):
root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- relevanceScore: string (nullable = true)
 |-- providerId: string (nullable = true)
 |-- providerHotelId: string (nullable = true)
 |-- providerName: string (nullable = true)
 |-- language: string (nullable = true)
 |-- geoCode: struct (nullable = true)
 |    |-- lat: string (nullable = true)
 |    |-- long: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- line1: string (nullable = true)
 |    |    |-- city: struct (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |-- state: struct (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |-- country: struct (nullable = true)
 |    |    |    |-- code: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |-- postalCode: string (nullable = true)
 |    |-- phones: array (nulla

In [29]:
# Ensure overall_pair_score is present — compute it if the source parquet predates the scorer update
if "overall_pair_score" not in df.columns:
    import sys

    sys.path.insert(0, "/Users/nakul.patil/Documents/hotel-mapping/src")
    from hotel_data.pipeline.scoring.scorers.overall_pair_scorer import (
        build_overall_pair_score_expr,
    )

    df = df.withColumn("overall_pair_score", build_overall_pair_score_expr())
    print("✓ Computed overall_pair_score via scorer expression")
else:
    print(
        "✓ overall_pair_score already present in hotel pair data — passing through to CSV"
    )

✓ overall_pair_score already present in hotel pair data — passing through to CSV


In [30]:
import os

output_path = "/Users/nakul.patil/Downloads/hotels.csv"

# Print all columns being written
print(f"Writing {len(df.columns)} columns to CSV:")
for c in df.columns:
    print(f"  {c}")

# Delete existing file before writing
if os.path.exists(output_path):
    os.remove(output_path)
    print(f"\n✓ Deleted existing file: {output_path}")

print(f"\nWriting data to: {output_path}")
pdf = df.toPandas()
pdf.to_csv(output_path, index=False)
print(f"✓ Successfully exported {len(pdf):,} pairs to {output_path}")

Writing 50 columns to CSV:
  id_i
  id_j
  uid_i
  uid_j
  providerName_i
  providerName_j
  providerHotelId_i
  providerHotelId_j
  name_i
  name_j
  normalized_name_i
  normalized_name_j
  type_i
  type_j
  geo_distance_km
  name_score_containment
  normalized_name_score_containment
  name_score_jaccard
  normalized_name_score_jaccard
  name_score_lcs
  normalized_name_score_lcs
  name_score_levenshtein
  normalized_name_score_levenshtein
  name_score_sbert
  normalized_name_score_sbert
  average_name_score
  average_normalized_name_score
  address_line1_score
  postal_code_match
  country_match
  address_sbert_score
  phone_match_score
  email_match_score
  fax_match_score
  property_type_score
  name_unit_score
  address_unit_score
  supplier_score
  star_ratings_score
  overall_pair_score
  starRating_i
  contact_address_line1_i
  contact_address_postalCode_i
  contact_address_city_name_i
  contact_address_state_name_i
  starRating_j
  contact_address_line1_j
  contact_address_pos